In [21]:
import numpy as np 
import tensorflow as tf 
import pandas as pd
from tensorflow import keras

## 1 - Notation

| General Notation      | Description                                                                 | Python (if any)   |
|:----------------------|:-----------------------------------------------------------------------------|:------------------|
| $r(i,j)$              | Scalar; = 1 if user $j$ rated movie $i$, = 0 otherwise                       | R[i, j]           |
| $y(i,j)$              | Scalar; rating given by user $j$ to movie $i$ (only if $r(i,j) = 1$)         | Y[i, j]           |
| $\mathbf{w}^{(j)}$    | Vector; parameters (preferences) for user $j$                                | W[j]              |
| $b^{(j)}$             | Scalar; bias term for user $j$                                               | b[0, j] or b[j]   |
| $\mathbf{x}^{(i)}$    | Vector; features of movie $i$                                                | X[i]              |
| $n_u$                 | Number of users                                                              | num_users         |
| $n_m$                 | Number of movies                                                             | num_movies        |
| $n$                   | Number of features                                                           | num_features      |
| $\mathbf{X}$          | Matrix; rows are $\mathbf{x}^{(i)}$ for all movies                           | X                 |
| $\mathbf{W}$          | Matrix; rows are $\mathbf{w}^{(j)}$ for all users                            | W                 |
| $\mathbf{b}$          | Vector of bias parameters $b^{(j)}$ for all users                            | b                 |
| $\mathbf{R}$          | Matrix; elements $r(i,j)$ indicating whether user $j$ rated movie $i$        | R                 |


## 2 - Recommender Systems 

The goal of a collaborative filtering system is to generate two vectors. For each user, a 'parameter vector' that embodies some description the movie tastes of a user. For each movie, a feature vector of the same size which embodies some description of the move. The dot product of the two vectors plus the bias term should produce an estimate of the rating the user might give to that movie.

Existing ratings are provided in matrix form as shown. Y contains ratings: 0.5 to 5 inclusive in 0.5 steps. 0 If the movie has not been rated, R has a 1 where movies have been rated. Movies are in rows, users are in columns. Eeach user has a parameter $w^{user}$ and bias. Each movie has a feature vector $x^{movie}$. There vectors are simultaneously learned by the existing user/movie ratings as training data. One training example is shown above: 
$\mathbf{w}^{(1)} \cdot \mathbf{x}^{(1)} + b^{(1)} = 4$. It is worth noting that the feature vector $x^{movie}$ must satisfy all the users while the user vector $w^{user}$ must satisfy all the movies.

Once the feature vectors and parameters are learned, they can be used to predict how a user might rate an unrated movie. The equation is an example of predicting a rating for user on movie zero. 

## 3 - Movie ratings dataset

In [6]:
def load_precalc_params_small(): 

    file = open('./data/small_movies_X.csv', 'rb') 
    X = np.loadtxt(file, delimiter = ',') 

    file = open('./data/small_movies_W.csv', 'rb') 
    W = np.loadtxt(file, delimiter = ',') 

    file = open('./data/small_movies_b.csv', 'rb') 
    b = np.loadtxt(file, delimiter = ',')

    b = b.reshape(1, -1) 
    num_movies, num_features = X.shape 
    num_users,_ = W.shape 

    return (X, W, b, num_movies, num_features, num_users) 

In [8]:
def load_ratings_small(): 
    file = open('./data/small_movies_Y.csv', 'rb') 
    Y = np.loadtxt(file, delimiter = ',') 

    file = open('./data/small_movies_R.csv', 'rb') 
    R = np.loadtxt(file, delimiter = ',') 

    return (Y, R) 

In [9]:
#Load data
X, W, b, num_movies, num_features, num_users = load_precalc_params_small()
Y, R = load_ratings_small()

print("Y", Y.shape, "R", R.shape)
print("X", X.shape)
print("W", W.shape)
print("b", b.shape)
print("num_features", num_features)
print("num_movies",   num_movies)
print("num_users",    num_users)

Y (4778, 443) R (4778, 443)
X (4778, 10)
W (443, 10)
b (1, 443)
num_features 10
num_movies 4778
num_users 443


In [11]:
# From the matrix, we can compute statistics like average rating 

tsmean = np.mean(Y[0, R[0, :].astype(bool)]) 
print(f"Average rating for movie 1: {tsmean:0.3f} / 5")

Average rating for movie 1: 3.400 / 5


## 4 - Collaborative filtering learning algorithm

Now, you will begin implementing the collaborative filtering learning
algorithm. You will start by implementing the objective function. 

The collaborative filtering algorithm in the setting of movie
recommendations considers a set of $n$-dimensional parameter vectors
$\mathbf{x}^{(0)},...,\mathbf{x}^{(n_m-1)}$, $\mathbf{w}^{(0)},...,\mathbf{w}^{(n_u-1)}$ and $b^{(0)},...,b^{(n_u-1)}$, where the
model predicts the rating for movie $i$ by user $j$ as
$y^{(i,j)} = \mathbf{w}^{(j)}\cdot \mathbf{x}^{(i)} + b^{(j)}$ . Given a dataset that consists of
a set of ratings produced by some users on some movies, you wish to
learn the parameter vectors $\mathbf{x}^{(0)},...,\mathbf{x}^{(n_m-1)},
\mathbf{w}^{(0)},...,\mathbf{w}^{(n_u-1)}$  and $b^{(0)},...,b^{(n_u-1)}$ that produce the best fit (minimizes
the squared error).

You will complete the code in cofiCostFunc to compute the cost
function for collaborative filtering. 

### 4.1 Collaborative filtering cost function

The collaborative filtering cost function is given by
$$J({\mathbf{x}^{(0)},...,\mathbf{x}^{(n_m-1)},\mathbf{w}^{(0)},b^{(0)},...,\mathbf{w}^{(n_u-1)},b^{(n_u-1)}})= \left[ \frac{1}{2}\sum_{(i,j):r(i,j)=1}(\mathbf{w}^{(j)} \cdot \mathbf{x}^{(i)} + b^{(j)} - y^{(i,j)})^2 \right]
+ \underbrace{\left[
\frac{\lambda}{2}
\sum_{j=0}^{n_u-1}\sum_{k=0}^{n-1}(\mathbf{w}^{(j)}_k)^2
+ \frac{\lambda}{2}\sum_{i=0}^{n_m-1}\sum_{k=0}^{n-1}(\mathbf{x}_k^{(i)})^2
\right]}_{regularization}
\tag{1}$$
The first summation in (1) is "for all $i$, $j$ where $r(i,j)$ equals $1$" and could be written:

$$
= \left[ \frac{1}{2}\sum_{j=0}^{n_u-1} \sum_{i=0}^{n_m-1}r(i,j)*(\mathbf{w}^{(j)} \cdot \mathbf{x}^{(i)} + b^{(j)} - y^{(i,j)})^2 \right]
+\text{regularization}
$$

You should now write cofiCostFunc (collaborative filtering cost function) to return this cost.

In [16]:
def cofi_cost_func(X, W, b, Y, R, lambda_): 
    """
    Returns the cost for the content-based filtering 
    Args: 
        X (ndarray (num_movies, num_features) : matrix of item features
        W (ndarray (num_users, num_features)  : matrix of user parameters
        b (ndarray(1, num_users) : vector of user parameters
        Y (ndarray(num_movies, num_users) : matrix of user ratings of movies 
        R (ndarray(num_movies, num_users) : matrix, where R(i, j) = 1 if the i-th movies was rated by the j-th user
        lambda_ (float) : regularization parameter 

    Returns: 
        J(float): cost 
    """

    nm, nu = Y.shape 
    J = 0

    pred = X @ W.T + b
    error = R * (pred - Y)**2 
    cost = 0.5 * np.sum(error) 

    reg_W = (lambda_ / 2) * np.sum(W**2)
    reg_X = (lambda_ / 2) * np.sum(X**2)

    J = cost + reg_W + reg_X 

    return J

In [17]:
# Reduce the data set size so that this runs faster
num_users_r = 4
num_movies_r = 5 
num_features_r = 3

X_r = X[:num_movies_r, :num_features_r]
W_r = W[:num_users_r,  :num_features_r]
b_r = b[0, :num_users_r].reshape(1,-1)
Y_r = Y[:num_movies_r, :num_users_r]
R_r = R[:num_movies_r, :num_users_r]

# Evaluate cost function
J = cofi_cost_func(X_r, W_r, b_r, Y_r, R_r, 0);
print(f"Cost: {J:0.2f}")

Cost: 13.67


In [18]:
# Evaluate cost function with regularization 
J = cofi_cost_func(X_r, W_r, b_r, Y_r, R_r, 1.5);
print(f"Cost (with regularization): {J:0.2f}")

Cost (with regularization): 28.09


In [23]:
def load_Movie_list_pd(): 
    df = pd.read_csv('./data/small_movie_list.csv', header = 0, index_col = 0, 
                     delimiter = ',', quotechar = '"') 
    mlist = df["title"].to_list()

    return (mlist, df)

In [24]:
movieList, movieList_df = load_Movie_list_pd()

In [25]:
my_ratings = np.zeros(num_movies) # Initialize my ratings

In [26]:
# Check the file small_movie_list.csv for id of each movie in our dataset
# For example, Toy Story 3 (2010) has ID 2700, so to rate it "5", you can set
my_ratings[2700] = 5 

#Or suppose you did not enjoy Persuasion (2007), you can set
my_ratings[2609] = 2;

# We have selected a few movies we liked / did not like and the ratings we
# gave are as follows:
my_ratings[929]  = 5   # Lord of the Rings: The Return of the King, The
my_ratings[246]  = 5   # Shrek (2001)
my_ratings[2716] = 3   # Inception
my_ratings[1150] = 5   # Incredibles, The (2004)
my_ratings[382]  = 2   # Amelie (Fabuleux destin d'Amélie Poulain, Le)
my_ratings[366]  = 5   # Harry Potter and the Sorcerer's Stone (a.k.a. Harry Potter and the Philosopher's Stone) (2001)
my_ratings[622]  = 5   # Harry Potter and the Chamber of Secrets (2002)
my_ratings[988]  = 3   # Eternal Sunshine of the Spotless Mind (2004)
my_ratings[2925] = 1   # Louis Theroux: Law & Disorder (2008)
my_ratings[2937] = 1   # Nothing to Declare (Rien à déclarer)
my_ratings[793]  = 5   # Pirates of the Caribbean: The Curse of the Black Pearl (2003)

In [27]:
my_rated = [i for i in range(len(my_ratings)) if my_ratings[i] > 0]

In [28]:
print('\nNew user ratings:\n') 
for i in range(len(my_ratings)): 
    if my_ratings[i] > 0:
        print(f'Rated {my_ratings[i]} for  {movieList_df.loc[i,"title"]}');


New user ratings:

Rated 5.0 for  Shrek (2001)
Rated 5.0 for  Harry Potter and the Sorcerer's Stone (a.k.a. Harry Potter and the Philosopher's Stone) (2001)
Rated 2.0 for  Amelie (Fabuleux destin d'Amélie Poulain, Le) (2001)
Rated 5.0 for  Harry Potter and the Chamber of Secrets (2002)
Rated 5.0 for  Pirates of the Caribbean: The Curse of the Black Pearl (2003)
Rated 5.0 for  Lord of the Rings: The Return of the King, The (2003)
Rated 3.0 for  Eternal Sunshine of the Spotless Mind (2004)
Rated 5.0 for  Incredibles, The (2004)
Rated 2.0 for  Persuasion (2007)
Rated 5.0 for  Toy Story 3 (2010)
Rated 3.0 for  Inception (2010)
Rated 1.0 for  Louis Theroux: Law & Disorder (2008)
Rated 1.0 for  Nothing to Declare (Rien à déclarer) (2010)


In [51]:
# Useful values
num_movies, num_users = Y.shape 
num_features = 100

tf.random.set_seed(1234) 
W = tf.Variable(tf.random.normal((num_users,  num_features),dtype=tf.float64),  name='W')
X = tf.Variable(tf.random.normal((num_movies, num_features),dtype=tf.float64),  name='X')
b = tf.Variable(tf.random.normal((1,          num_users),   dtype=tf.float64),  name='b')

# Instantiate an optimizer.
optimizer = keras.optimizers.Adam(learning_rate=1e-1)

The operations involved in learning $w$, $b$, and $x$ simultaneously do not fall into the typical 'layers' offered in the TensorFlow neural network package.  Consequently, the flow used in Course 2: Model, Compile(), Fit(), Predict(), are not directly applicable. Instead, we can use a custom training loop.

Recall from earlier labs the steps of gradient descent.
- repeat until convergence:
    - compute forward pass
    - compute the derivatives of the loss relative to parameters
    - update the parameters using the learning rate and the computed derivatives 
    
TensorFlow has the marvelous capability of calculating the derivatives for you. This is shown below. Within the `tf.GradientTape()` section, operations on Tensorflow Variables are tracked. When `tape.gradient()` is later called, it will return the gradient of the loss relative to the tracked variables. The gradients can then be applied to the parameters using an optimizer. 
This is a very brief introduction to a useful feature of TensorFlow and other machine learning frameworks. Further information can be found by investigating "custom training loops" within the framework of interest.

In [40]:
def normalizeRatings(Y, R): 
    """
    Preprocess data by subtracting mean rating for every movie (every row). 
    Only include real ratings R(i, j) = 1
    """
    Ymean = (np.sum(Y*R, axis=1) / (np.sum(R, axis=1)+1e-12)).reshape(-1, 1) 
    Ynorm = Y - np.multiply(Ymean, R) 
    return (Ynorm, Ymean) 

In [50]:
Y, R = load_ratings_small()

Y = np.c_[my_ratings, Y] 
R = np.c_[(my_ratings != 0).astype(int), R] 

Ynorm, Ymean = normalizeRatings(Y, R) 

In [45]:
def cofi_cost_func_v(X, W, b, Y, R, lambda_):
    """
    Returns the cost for the content-based filtering
    Vectorized for speed. Uses tensorflow operations to be compatible with custom training loop.
    Args:
      X (ndarray (num_movies,num_features)): matrix of item features
      W (ndarray (num_users,num_features)) : matrix of user parameters
      b (ndarray (1, num_users)            : vector of user parameters
      Y (ndarray (num_movies,num_users)    : matrix of user ratings of movies
      R (ndarray (num_movies,num_users)    : matrix, where R(i, j) = 1 if the i-th movies was rated by the j-th user
      lambda_ (float): regularization parameter
    Returns:
      J (float) : Cost
    """
    j = (tf.linalg.matmul(X, tf.transpose(W)) + b - Y)*R
    J = 0.5 * tf.reduce_sum(j**2) + (lambda_/2) * (tf.reduce_sum(X**2) + tf.reduce_sum(W**2))
    return J

In [52]:
iterations = 200 
lambda_ = 1 

for iter in range(iterations): 

    with tf.GradientTape() as tape: 

        cost_value = cofi_cost_func_v(X, W, b, Ynorm, R, lambda_) 

    grads = tape.gradient(cost_value, [X, W, b]) 

    optimizer.apply_gradients(zip(grads, [X, W, b])) 

    if iter % 20 == 0: 
        print(f"Training loss at iteration {iter}: {cost_value:0.1f}")

Training loss at iteration 0: 2321191.3
Training loss at iteration 20: 136169.3
Training loss at iteration 40: 51863.7
Training loss at iteration 60: 24599.0
Training loss at iteration 80: 13630.6
Training loss at iteration 100: 8487.7
Training loss at iteration 120: 5807.8
Training loss at iteration 140: 4311.6
Training loss at iteration 160: 3435.3
Training loss at iteration 180: 2902.1


In [56]:
p = np.matmul(X.numpy(), np.transpose(W.numpy()) + b.numpy())

pm = p + Ymean

In [57]:
my_predictions = pm[:,0]

# sort predictions
ix = tf.argsort(my_predictions, direction='DESCENDING')

for i in range(17):
    j = ix[i]
    if j not in my_rated:
        print(f'Predicting rating {my_predictions[j]:0.2f} for movie {movieList[j]}')

print('\n\nOriginal vs Predicted ratings:\n')
for i in range(len(my_ratings)):
    if my_ratings[i] > 0:
        print(f'Original {my_ratings[i]}, Predicted {my_predictions[i]:0.2f} for {movieList[i]}')

Predicting rating 8.12 for movie Crouching Tiger, Hidden Dragon (Wo hu cang long) (2000)
Predicting rating 7.01 for movie The Hunger Games (2012)
Predicting rating 6.92 for movie Hero (Ying xiong) (2002)
Predicting rating 6.76 for movie Inside Man (2006)
Predicting rating 6.72 for movie Spider-Man 2 (2004)
Predicting rating 6.60 for movie Finding Nemo (2003)
Predicting rating 6.52 for movie Dark Knight, The (2008)
Predicting rating 6.41 for movie Shallow Hal (2001)
Predicting rating 6.39 for movie X-Men (2000)
Predicting rating 6.13 for movie Lost in Translation (2003)
Predicting rating 6.07 for movie Kill Bill: Vol. 2 (2004)
Predicting rating 6.04 for movie Meet the Parents (2000)
Predicting rating 5.98 for movie Lord of the Rings: The Fellowship of the Ring, The (2001)
Predicting rating 5.90 for movie Hangover, The (2009)
Predicting rating 5.85 for movie Road to Perdition (2002)
Predicting rating 5.80 for movie X2: X-Men United (2003)
Predicting rating 5.79 for movie Zombieland (2009

In [58]:
filter=(movieList_df["number of ratings"] > 20)
movieList_df["pred"] = my_predictions
movieList_df = movieList_df.reindex(columns=["pred", "mean rating", "number of ratings", "title"])
movieList_df.loc[ix[:300]].loc[filter].sort_values("mean rating", ascending=False)

,pred,mean rating,number of ratings,title
4553,4.841774,4.280000,25,Logan (2017)
1743,4.593587,4.252336,107,"Departed, The (2006)"
2112,6.522799,4.238255,149,"Dark Knight, The (2008)"
5,4.947386,4.220930,43,"Boondock Saints, The (2000)"
382,5.035357,4.183333,120,"Amelie (Fabuleux destin d'Amélie Poulain, Le) ..."
...,...,...,...,...
1282,5.646385,3.000000,61,Charlie and the Chocolate Factory (2005)
2627,5.003872,2.875000,28,Alice in Wonderland (2010)
2401,5.640932,2.865385,26,X-Men Origins: Wolverine (2009)
788,5.250453,2.560606,33,Hulk (2003)
